In [1]:
# ====================================================
# Unified model comparison table (pandas)
# ====================================================

import pandas as pd
from scipy.stats import chi2 as chi2_dist

# ---- Stored results ----
models = {
    "LCDM": {"lnL": -1383.3235, "k": 6,  "AIC": 2778.6470, "BIC": 2811.4168},
    "wCDM": {"lnL": -1378.0069, "k": 7,  "AIC": 2770.0137, "BIC": 2808.2452},
    "CPL":  {"lnL": -1377.7934, "k": 8,  "AIC": 2771.5868, "BIC": 2815.2799},
    "AB":   {"lnL": -1383.4515, "k": 7,  "AIC": 2780.9029, "BIC": 2819.1344},
    "HS":   {"lnL": -1378.2402, "k": 7,  "AIC": 2770.4804, "BIC": 2808.7119},
    "ST":   {"lnL": -1378.2307, "k": 7,  "AIC": 2770.4615, "BIC": 2808.6930},
}

N_tot = 1740
ref   = "LCDM"

# ---- Compute derived quantities ----
for name, m in models.items():
    m["chi2"]     = -2.0 * m["lnL"]
    m["chi2_red"] = m["chi2"] / (N_tot - m["k"])

    if name == ref:
        m["delta_lnL"] = None
        m["delta_AIC"] = None
        m["delta_BIC"] = None
        m["delta_k"]   = None
        m["lnBF"]      = None
        m["p_value"]   = None
        m["AIC_verd"]  = "Reference"
        m["BIC_verd"]  = "Reference"
        m["LRT_verd"]  = "Reference"
    else:
        m["delta_lnL"] = m["lnL"] - models[ref]["lnL"]
        m["delta_AIC"] = m["AIC"] - models[ref]["AIC"]
        m["delta_BIC"] = m["BIC"] - models[ref]["BIC"]
        m["delta_k"]   = m["k"]   - models[ref]["k"]
        # ln Bayes Factor approximation via BIC:
        # ln B_10 ~ -0.5 * delta_BIC  (positive = favors alternative)
        m["lnBF"]      = -0.5 * m["delta_BIC"]
        m["p_value"]   = chi2_dist.sf(2.0 * m["delta_lnL"], df=m["delta_k"])


        # ---- AIC and BIC verdicts: Plaza & Kraiselburd (2025) ----
        
        # delta_AIC = AIC_alt - AIC_ref
        d = m["delta_AIC"]
        if   abs(d) <= 2:  m["AIC_verd"] = "Weak"
        elif abs(d) <= 6:  m["AIC_verd"] = "Positive for alternative" if d < 0 else "Positive for reference"
        elif abs(d) <= 10: m["AIC_verd"] = "Strong for alternative"   if d < 0 else "Strong for reference"
        else:              m["AIC_verd"] = "Very strong for alternative" if d < 0 else "Very strong for reference"
        
        # delta_BIC = BIC_alt - BIC_ref
        b = m["delta_BIC"]
        if   abs(b) <= 2:  m["BIC_verd"] = "Weak"
        elif abs(b) <= 6:  m["BIC_verd"] = "Positive for alternative" if b < 0 else "Positive for reference"
        elif abs(b) <= 10: m["BIC_verd"] = "Strong for alternative"   if b < 0 else "Strong for reference"
        else:              m["BIC_verd"] = "Very strong for alternative" if b < 0 else "Very strong for reference"

        # ---- LRT verdict ----
        p = m["p_value"]
        if   p < 0.001: m["LRT_verd"] = "p < 0.001  highly significant"
        elif p < 0.01:  m["LRT_verd"] = f"p = {p:.4f}  significant"
        elif p < 0.05:  m["LRT_verd"] = f"p = {p:.4f}  significant"
        else:           m["LRT_verd"] = f"p = {p:.4f}  not significant"

# ---- Helper formatters ----
def fmt(val, spec=".4f"):
    if val is None: return "—"
    if isinstance(val, float): return format(val, spec)
    return str(val)

def fmt_signed(val, spec=".4f"):
    if val is None: return "—"
    return f"{val:+{spec}}"

# ---- Build DataFrames ----
names = list(models.keys())

df_stats = pd.DataFrame({
    "Quantity": ["k", "ln L_max", "chi2", "chi2/dof", "AIC", "BIC"],
    **{n: [
        fmt(models[n]["k"], "d"),
        fmt(models[n]["lnL"]),
        fmt(models[n]["chi2"]),
        fmt(models[n]["chi2_red"]),
        fmt(models[n]["AIC"]),
        fmt(models[n]["BIC"]),
    ] for n in names}
}).set_index("Quantity")

df_diff = pd.DataFrame({
    "Quantity": ["delta_k", "Delta ln L", "Delta AIC", "Delta BIC", "ln BF"],
    **{n: [
        fmt(models[n]["delta_k"], "d"),
        fmt_signed(models[n]["delta_lnL"]),
        fmt_signed(models[n]["delta_AIC"]),
        fmt_signed(models[n]["delta_BIC"]),
        fmt_signed(models[n]["lnBF"]),
    ] for n in names}
}).set_index("Quantity")

df_verd = pd.DataFrame({
    "Quantity": ["AIC verdict", "BIC verdict", "LRT verdict"],
    **{n: [
        models[n]["AIC_verd"],
        models[n]["BIC_verd"],
        models[n]["LRT_verd"],
    ] for n in names}
}).set_index("Quantity")

# ---- pandas display settings ----
pd.set_option("display.max_rows",     None)
pd.set_option("display.max_columns",  None)
pd.set_option("display.width",        140)
pd.set_option("display.max_colwidth", 40)

# ---- Save to file ----
output_path = "model_comparison.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n" + "=" * 80 + "\n")
    f.write(f"{'UNIFIED MODEL COMPARISON  (CC + SNe + BAO + FRB)':^80}\n")
    f.write(f"{'Reference: LCDM':^80}\n")
    f.write("=" * 80 + "\n")

    f.write("\n  --- Statistics ---\n")
    f.write(df_stats.to_string() + "\n")

    f.write("\n" + "-" * 80 + "\n")
    f.write("\n  --- Differences vs LCDM ---\n")
    f.write(df_diff.to_string() + "\n")

    f.write("\n" + "-" * 80 + "\n")
    f.write("\n  --- Verdicts ---\n")
    f.write(df_verd.to_string() + "\n")

    f.write("\n" + "=" * 80 + "\n")

print(f"Table saved in: {output_path}")

Table saved in: model_comparison.txt
